# **Using MRI for Brain Tumor Detection and Segmentation**

**Authors:** Isha Dev, Dharvi Jagirdar, Rohan Giridharan

This notebook contains the code to form the LLM prompt and images, and generate the LLM outputs from `Radiology Infer-Mini` LLM. Following is the use of Sentence Transformer `all-MiniLM-L6-v2` to generate embedding similarity scores.

## **Set Up and Imports**

In [17]:
!pip install -q qwen_vl_utils
!pip install -q sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 40.1 MB/s eta 0:00:00


In [18]:
import os
import pandas as pd
import numpy as np
import re
import torch
from tqdm import tqdm
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from sentence_transformers import SentenceTransformer, util
import qwen_vl_utils
from qwen_vl_utils import process_vision_info


from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "prithivMLmods/Radiology-Infer-Mini",
    device_map="auto",              # Automatically puts layers on GPU if available
    torch_dtype=torch.float16       # Recommended for GPU memory efficiency (A100, V100, etc.)
).eval()                            # Set model to evaluation mode

processor = AutoProcessor.from_pretrained(
    "prithivMLmods/Radiology-Infer-Mini",
    trust_remote_code=True,
    size={"shortest_edge": 512, "longest_edge": 1792},
    min_pixels=512 * 512,
    max_pixels=1280 * 1280
)

config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/4.42G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/408 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

In [60]:
# Fix center_voxel for malformed string formatting
def safe_parse_center_voxel(s):
    matches = re.findall(r'\d+', str(s))
    return tuple(map(int, matches))

# Function to form LLM input
def format_llm_input(case_id, test_results_dict, t1ce_img_path, pred_img_path):
    """
    Builds a string + image input for the LLM.
    case_id: e.g., "306"
    test_results_dict: the test_results list or dict with 'case' key
    gt_img: path to GT montage image
    pred_img: path to predicted montage image
    """
    # Get info from test_results
    result = next(r for r in test_results_dict if r['case'] == case_id)

    def format_section(section_dict):
        return "\n".join([f"- {k}: {v}" for k, v in section_dict.items()])

    # Build prompt
    prompt = f"""
Patient ID: {case_id}

== GROUND TRUTH SEGMENTATION ==
Tumor Volumes:
{format_section(result['ground_truth']['volumes'])}

Presence of Classes:
{format_section(result['ground_truth']['presence'])}

Anatomical Location:
{format_section(result['ground_truth']['location'])}

Region Summary:
{format_section(result['ground_truth']['summary'])}

Radiological Features:
{format_section(result['ground_truth']['radiology_features'])}

== PREDICTED SEGMENTATION ==
Tumor Volumes:
{format_section(result['prediction']['volumes'])}

Presence of Classes:
{format_section(result['prediction']['presence'])}

Anatomical Location:
{format_section(result['prediction']['location'])}

Region Summary:
{format_section(result['prediction']['summary'])}

Radiological Features:
{format_section(result['prediction']['radiology_features'])}

Attached are the 3D reconstructions (flattened montage) of both the ground truth and predicted segmentations.

Please write a detailed clinical report as if you are preparing a case summary for a multidisciplinary team (radiologist, oncologist, and neurologist). The report should include:
1. An overview of the tumor including anatomical location and laterality, approximate center of mass and volume distribution and morphological characteristics.
2. Tumor class description including the core/non-enhancing, edema and enhancing tumor regions.
3. CLinical interpretation including possible diagnosis based on segmentation patterns and potential treatment options
4. Prognostic insight included estimated survival probabilities or literature-based prognosis related to the tumor presentation
5. Literatures that may aid in tumor diagnosis and patient treatment
6. Recommended next steps including further imaging or biopsy needs and interventions based on segmentation.

Write this as a structured medical report that would be shared in a tumore board review. Assume this is one of multiple patients being reviewed for surgical planning or oncological intervention.
"""
    return prompt, t1ce_img_path, pred_img_path

## **AUNet** LLM Generation

In [61]:
# Get test subject summaries saved from segmentation model
subject_data = pd.read_csv('/content/drive/MyDrive/AUNet/test_results_summary.csv')
subject_ids = subject_data['case'].tolist()

montage_path = '/content/drive/MyDrive/AUNet/Montages'

In [62]:
# Fix center voxel error
subject_data["center_voxel"] = subject_data["center_voxel"].apply(safe_parse_center_voxel)
subject_data["core_com"] = subject_data["core_com"].apply(safe_parse_center_voxel)
subject_data["edema_com"] = subject_data["edema_com"].apply(safe_parse_center_voxel)
subject_data["enhancing_com"] = subject_data["enhancing_com"].apply(safe_parse_center_voxel)

# Build structured test results
test_results = []
for case, grp in subject_data.groupby("case", sort=False):
    spacing = grp[["spacing_x", "spacing_y", "spacing_z"]].iloc[0].tolist()
    entry = {"case": case, "spacing_mm": spacing}

    for mode, sub in grp.groupby("type"):
        row = sub.iloc[0]
        entry[mode] = {
            "volumes": {
                "core": {"volume_mm3": row["vol_core"]},
                "edema": {"volume_mm3": row["vol_edema"]},
                "enhancing": {"volume_mm3": row["vol_enhancing"]},
                "total": {"volume_mm3": row["vol_total"]},
            },
            "presence": {
                "core": bool(row["has_core"]),
                "edema": bool(row["has_edema"]),
                "enhancing": bool(row["has_enhancing"]),
            },
            "location": {
                "hemisphere": row["hemisphere"],
                "approx_center_voxel": row["center_voxel"]
            },
            "radiology_features": {
                "mass_effect": row["mass_effect"],
                "contrast_enhancement": row["contrast_enhancement"],
                "irregularity": row["irregularity"],
                "laterality": row["laterality"],
            },
            "summary": {
                "core": {
                    "components": row["core_components"],
                    "center_of_mass": row["core_com"],
                    "max_cross_sectional_area_voxels": row["core_max_area"],
                },
                "edema": {
                    "components": row["edema_components"],
                    "center_of_mass": row["edema_com"],
                    "max_cross_sectional_area_voxels": row["edema_max_area"],
                },
                "enhancing": {
                    "components": row["enhancing_components"],
                    "center_of_mass": row["enhancing_com"],
                    "max_cross_sectional_area_voxels": row["enhancing_max_area"],
                },
            },
        }
    test_results.append(entry)

In [63]:
# Build list of formatted prompts with paths
llm_prompt_data = []

for result in test_results:
    case_id = result["case"]
    case_id_str = str(case_id).zfill(3)  # Ensure 3-digit format

    t1ce_img_path = os.path.join(montage_path, f"{case_id_str}_t1ce_montage.png")
    pred_img_path = os.path.join(montage_path, f"{case_id_str}_prediction_montage.png")

    # Add keys to match expected input
    result_formatted = {
        "case": case_id,
        "ground_truth": result.get("ground_truth", {}),
        "prediction": result.get("prediction", {})
    }

    # Format prompt
    try:
        prompt_text, gt_path, pred_path = format_llm_input(case_id, [result_formatted], t1ce_img_path, pred_img_path)
    except Exception as e:
        print(f"Skipping case {case_id} due to error: {e}")
        continue

    llm_prompt_data.append({
        "case": case_id,
        "prompt": prompt_text.strip(),
        "t1ce_montage_path": gt_path,
        "pred_montage_path": pred_path
    })

# Save
llm_prompts_df = pd.DataFrame(llm_prompt_data)
llm_prompts_df.to_csv('/content/drive/MyDrive/AUNet/AUNet_LLM_Prompts.csv', index=False)
print("Saved all results to AUNet_LLM_Prompts.csv")

Saved all results to AUNet_LLM_Prompts.csv


In [64]:
# Get the first entry from the dataframe
entry = llm_prompts_df.iloc[0]
llm_prompt = entry['prompt'].replace("nan", "not reported")
t1ce_img_path = entry['t1ce_montage_path']
pred_img_path = entry['pred_montage_path']

# Load both images
t1ce_img_path = Image.open(t1ce_img_path).convert("RGB")
pred_img = Image.open(pred_img_path).convert("RGB")

# Create the LLM message format with both images
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": gt_img},
            {"type": "image", "image": pred_img},
            {"type": "text", "text": llm_prompt}
        ],
    }
]

# Preprocess prompt and images
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)

# Tokenize inputs
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
)

# Send to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = inputs.to(device)

# Generate response
generated_ids = model.generate(
    **inputs,
    max_new_tokens=2048,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.2
)

# Decode only the new tokens
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

# Print the LLM output
print(output_text[0])

Case Report
This case presents with an MRI scan showing a left-sided brain lesion with heterogeneous enhancement pattern. The lesion has been classified into three distinct components - core, edema, and enhancing areas. The core component comprises approximately 9% of the entire tumor volume while the edema component accounts for about 15%. The enhancing area represents less than 1%.

The core region shows non-necrotizing features such as irregular margins and focal calcifications. The surrounding edematous tissue appears hyperintense on T2-weighted images, indicating significant fluid accumulation within the lesion. The enhancing component consists mainly of blood vessels and hemosiderin deposits, which can be seen as bright spots on postcontrast sequences.

Clinical Presentation
A 22-year-old male presented with progressive neurological symptoms over several months. He had no prior history suggestive of malignancy but was found to have a left-sided intracranial mass during routine sc

In [65]:
# Output list to collect results
llm_outputs = []

# Eval mode
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Iterate over all cases
for idx, row in tqdm(llm_prompts_df.iterrows(), total=len(llm_prompts_df)):
    case_id = row['case']
    llm_prompt = row['prompt'].replace("nan", "not reported")
    t1ce_img_path = row['t1ce_montage_path']
    pred_img_path = row['pred_montage_path']

    try:
        # Load images
        gt_img = Image.open(t1ce_img_path).convert("RGB")
        pred_img = Image.open(pred_img_path).convert("RGB")

        # Construct LLM message
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": gt_img},
                    {"type": "image", "image": pred_img},
                    {"type": "text", "text": llm_prompt}
                ],
            }
        ]

        # Preprocess input
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to(device)

        # Generate response
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=2048,
            temperature=0.8,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2
        )

        # Decode
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )[0]

    except Exception as e:
        print(f"[{case_id}] Error: {e}")
        output_text = "ERROR: " + str(e)

    # Store results
    llm_outputs.append({
        "case": case_id,
        "prompt": llm_prompt,
        "llm_output": output_text
    })

# Save to CSV
llm_output_df = pd.DataFrame(llm_outputs)

100%|██████████| 33/33 [06:01<00:00, 10.96s/it]


In [66]:
llm_output_df.head()

,case,prompt,llm_output
0,222,Patient ID: 222\n\n== GROUND TRUTH SEGMENTATIO...,Case Report\nThis case presents with an MRI sc...
1,210,Patient ID: 210\n\n== GROUND TRUTH SEGMENTATIO...,Case Report\nThis case presents with an MRI sc...
2,26,Patient ID: 26\n\n== GROUND TRUTH SEGMENTATION...,Case Report\nThis case presents with an MRI sc...
3,125,Patient ID: 125\n\n== GROUND TRUTH SEGMENTATIO...,Case Report\nThis case presents with an MRI sc...
4,143,Patient ID: 143\n\n== GROUND TRUTH SEGMENTATIO...,Case Report\nThis case presents an MRI scan fr...


In [67]:
# Load model
model2 = SentenceTransformer('all-MiniLM-L6-v2')

# Embedding similarity metric
def compute_similarity(prompt_text, llm_output):
    embeddings = model2.encode([prompt_text, llm_output], convert_to_tensor=True)
    similarity = util.cos_sim(embeddings[0], embeddings[1]).item()
    return similarity

# Get scores
for idx, row in llm_output_df.iterrows():
    similarity = compute_similarity(row['prompt'], row['llm_output'])
    llm_output_df.loc[idx, 'embedding_similarity'] = similarity

In [68]:
llm_output_df.to_csv('/content/drive/MyDrive/AUNet/AUNet_LLM_Outputs.csv', index=False)
print("Saved all results to AUNet_LLM_Outputs.csv")

Saved all results to AUNet_LLM_Outputs.csv


In [76]:
mean_similarity = llm_output_df['embedding_similarity'].mean()
print(f"Mean embedding similarity score: {mean_similarity:.4f}")

Mean embedding similarity score: 0.3434


## **AUNet_Synthetic** LLM Generation

In [69]:
# Get test subject summaries saved from segmentation model
subject_data = pd.read_csv('/content/drive/MyDrive/AUNet_Synthetic/test_results_summary.csv')
subject_ids = subject_data['case'].tolist()

montage_path = '/content/drive/MyDrive/AUNet_Synthetic/Montages'

In [70]:
# Fix center voxel error
subject_data["center_voxel"] = subject_data["center_voxel"].apply(safe_parse_center_voxel)
subject_data["core_com"] = subject_data["core_com"].apply(safe_parse_center_voxel)
subject_data["edema_com"] = subject_data["edema_com"].apply(safe_parse_center_voxel)
subject_data["enhancing_com"] = subject_data["enhancing_com"].apply(safe_parse_center_voxel)

# Build structured test results
test_results = []
for case, grp in subject_data.groupby("case", sort=False):
    spacing = grp[["spacing_x", "spacing_y", "spacing_z"]].iloc[0].tolist()
    entry = {"case": case, "spacing_mm": spacing}

    for mode, sub in grp.groupby("type"):
        row = sub.iloc[0]
        entry[mode] = {
            "volumes": {
                "core": {"volume_mm3": row["vol_core"]},
                "edema": {"volume_mm3": row["vol_edema"]},
                "enhancing": {"volume_mm3": row["vol_enhancing"]},
                "total": {"volume_mm3": row["vol_total"]},
            },
            "presence": {
                "core": bool(row["has_core"]),
                "edema": bool(row["has_edema"]),
                "enhancing": bool(row["has_enhancing"]),
            },
            "location": {
                "hemisphere": row["hemisphere"],
                "approx_center_voxel": row["center_voxel"]
            },
            "radiology_features": {
                "mass_effect": row["mass_effect"],
                "contrast_enhancement": row["contrast_enhancement"],
                "irregularity": row["irregularity"],
                "laterality": row["laterality"],
            },
            "summary": {
                "core": {
                    "components": row["core_components"],
                    "center_of_mass": row["core_com"],
                    "max_cross_sectional_area_voxels": row["core_max_area"],
                },
                "edema": {
                    "components": row["edema_components"],
                    "center_of_mass": row["edema_com"],
                    "max_cross_sectional_area_voxels": row["edema_max_area"],
                },
                "enhancing": {
                    "components": row["enhancing_components"],
                    "center_of_mass": row["enhancing_com"],
                    "max_cross_sectional_area_voxels": row["enhancing_max_area"],
                },
            },
        }
    test_results.append(entry)

In [71]:
# Build list of formatted prompts with paths
llm_prompt_data = []

for result in test_results:
    case_id = result["case"]
    case_id_str = str(case_id).zfill(3)  # Ensure 3-digit format

    t1ce_img_path = os.path.join(montage_path, f"{case_id_str}_t1ce_montage.png")
    pred_img_path = os.path.join(montage_path, f"{case_id_str}_prediction_montage.png")

    # Add keys to match expected input
    result_formatted = {
        "case": case_id,
        "ground_truth": result.get("ground_truth", {}),
        "prediction": result.get("prediction", {})
    }

    # Format prompt
    try:
        prompt_text, gt_path, pred_path = format_llm_input(case_id, [result_formatted], t1ce_img_path, pred_img_path)
    except Exception as e:
        print(f"Skipping case {case_id} due to error: {e}")
        continue

    llm_prompt_data.append({
        "case": case_id,
        "prompt": prompt_text.strip(),
        "t1ce_montage_path": gt_path,
        "pred_montage_path": pred_path
    })

# Save
llm_prompts_df = pd.DataFrame(llm_prompt_data)
llm_prompts_df.to_csv('/content/drive/MyDrive/AUNet_Synthetic/AUNet_Synthetic_LLM_Prompts.csv', index=False)
print("Saved all results to AUNet_Synthetic_LLM_Prompts.csv")

Saved all results to AUNet_Synthetic_LLM_Prompts.csv


In [72]:
# Get the first entry from the dataframe
entry = llm_prompts_df.iloc[0]
llm_prompt = entry['prompt'].replace("nan", "not reported")
t1ce_img_path = entry['t1ce_montage_path']
pred_img_path = entry['pred_montage_path']

# Load both images
t1ce_img_path = Image.open(t1ce_img_path).convert("RGB")
pred_img = Image.open(pred_img_path).convert("RGB")

# Create the LLM message format with both images
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": gt_img},
            {"type": "image", "image": pred_img},
            {"type": "text", "text": llm_prompt}
        ],
    }
]

# Preprocess prompt and images
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)

# Tokenize inputs
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
)

# Send to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = inputs.to(device)

# Generate response
generated_ids = model.generate(
    **inputs,
    max_new_tokens=2048,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.2
)

# Decode only the new tokens
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

# Print the LLM output
print(output_text[0])

Case Report
This case presents an MRI scan from a right hemispheric glioblastoma multiforme with extensive edema and enhancement. The patient has undergone resection surgery previously but recurrence was noted at the time point of this study. This image shows the extent of residual disease after previous therapy.


In [73]:
# Output list to collect results
llm_outputs = []

# Eval mode
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Iterate over all cases
for idx, row in tqdm(llm_prompts_df.iterrows(), total=len(llm_prompts_df)):
    case_id = row['case']
    llm_prompt = row['prompt'].replace("nan", "not reported")
    t1ce_img_path = row['t1ce_montage_path']
    pred_img_path = row['pred_montage_path']

    try:
        # Load images
        gt_img = Image.open(t1ce_img_path).convert("RGB")
        pred_img = Image.open(pred_img_path).convert("RGB")

        # Construct LLM message
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": gt_img},
                    {"type": "image", "image": pred_img},
                    {"type": "text", "text": llm_prompt}
                ],
            }
        ]

        # Preprocess input
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to(device)

        # Generate response
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=2048,
            temperature=0.8,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2
        )

        # Decode
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )[0]

    except Exception as e:
        print(f"[{case_id}] Error: {e}")
        output_text = "ERROR: " + str(e)

    # Store results
    llm_outputs.append({
        "case": case_id,
        "prompt": llm_prompt,
        "llm_output": output_text
    })

# Save
llm_output_df_2 = pd.DataFrame(llm_outputs)

100%|██████████| 44/44 [08:42<00:00, 11.88s/it]


In [74]:
# Get scores
for idx, row in llm_output_df_2.iterrows():
    similarity = compute_similarity(row['prompt'], row['llm_output'])
    llm_output_df_2.loc[idx, 'embedding_similarity'] = similarity

llm_output_df_2.to_csv('/content/drive/MyDrive/AUNet_Synthetic/AUNet_Synthetic_LLM_Outputs.csv', index=False)
print("Saved all results to AUNet_Synthetic_LLM_Outputs.csv")

Saved all results to AUNet_Synthetic_LLM_Outputs.csv


In [75]:
mean_similarity = llm_output_df_2['embedding_similarity'].mean()
print(f"Mean embedding similarity score: {mean_similarity:.4f}")

Mean embedding similarity score: 0.3499
